# Fix Morphology — Batch

Post-processor for every `*_Morphology.csv` produced by notebook 14
(`Konversi dari Sumber Daya NLP (Cleaned) ke Leksikon Bilingual dan Kamus
Morfologi`).

**Problem.** The original `getMorfology()` routine stores the derived form
as a single unsplit string like `"mengabaikan vt melalekan"` and leaves the
`tag` column empty. Measured on dictionary #4 (Jambi), this affects ~43% of
rows.

**What this notebook does.**
1. Scans every morphology CSV to **learn** OCR-mangled POS tokens from
   corpus frequency (not hardcoded).
2. Splits `form` on the inline POS tag (`v`, `vt`, `vi`, `n`, `a`, `adv`,
   `num`, `pron`, `p`) plus any learned OCR-repairs.
3. Uses `LookupIsFromIndonesia.csv` to decide which half is Indonesian,
   so the fix works uniformly for both Indonesian→Regional and
   Regional→Indonesian dictionaries.
4. Writes cleaned copies to `../Ekstraksi/10. Morphology - Fixed/`. The
   original directory stays untouched so downstream notebooks still work.

Per dictionary you get two files:
- `<id>_Morphology.csv` — pipeline-compatible, same schema as before
- `<id>_Morphology_audit.csv` — adds `form_ind`, `form_reg`, `form_raw`,
  `ocr_pos_fixed`, `direction` for evaluation/research use.

## 1. Imports & paths

In [10]:
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional

import pandas as pd

# Shared helpers (direction lookup, POS regex, filename parsing)
from _common import (
    POS_TAGS,
    POS_SPLIT_RE,
    parse_dict_id,
    load_direction_lookup,
    direction_for,
    roles_for_morphology,
)

SRC_DIR = Path("../Ekstraksi/10. Morphology")
DST_DIR = Path("../Ekstraksi/10. Morphology - Fixed")
FIXUP_REPORT = DST_DIR / "ocr_pos_fixups.csv"

assert SRC_DIR.exists(), f"Source dir not found: {SRC_DIR.resolve()}"
DST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading from: {SRC_DIR.resolve()}")
print(f"Writing to:   {DST_DIR.resolve()}")

Reading from: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\10. Morphology
Writing to:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\10. Morphology - Fixed


## 2. OCR-aware edit distance

Standard edit distance treats every substitution as cost 1.0. But for OCR
repair, some substitutions are much more likely than others (`w` for `v`,
`n` for `m`, `i` for `l`). We give common OCR confusions a reduced cost of
0.5 so that close-but-visually-similar candidates beat unrelated ones.

This replaces the Damerau–Levenshtein distance used in the initial draft.

In [11]:
# Character pairs OCR commonly confuses in dictionary scans. Symmetric.
OCR_CONFUSION_PAIRS = {
    frozenset("vw"), frozenset("vy"), frozenset("nm"), frozenset("ih"),
    frozenset("il"), frozenset("rt"), frozenset("cl"), frozenset("oe"),
    frozenset("ua"), frozenset("hb"), frozenset("fl"),
}


def ocr_aware_distance(garbled: str, canonical: str) -> float:
    """Edit distance where OCR-confused character substitutions cost 0.5."""
    a, b = garbled.lower(), canonical.lower()
    la, lb = len(a), len(b)
    dp = [[0.0] * (lb + 1) for _ in range(la + 1)]
    for i in range(la + 1):
        dp[i][0] = float(i)
    for j in range(lb + 1):
        dp[0][j] = float(j)
    for i in range(1, la + 1):
        for j in range(1, lb + 1):
            if a[i - 1] == b[j - 1]:
                sub_cost = 0.0
            elif frozenset((a[i - 1], b[j - 1])) in OCR_CONFUSION_PAIRS:
                sub_cost = 0.5
            else:
                sub_cost = 1.0
            dp[i][j] = min(
                dp[i - 1][j] + 1.0,
                dp[i][j - 1] + 1.0,
                dp[i - 1][j - 1] + sub_cost,
            )
    return dp[la][lb]


# Sanity check
for pos in POS_TAGS:
    print(f"  w  vs {pos:<5} -> {ocr_aware_distance('w', pos)}")

  w  vs adv   -> 2.5
  w  vs pron  -> 4.0
  w  vs num   -> 3.0
  w  vs vt    -> 1.5
  w  vs vi    -> 1.5
  w  vs v     -> 0.5
  w  vs a     -> 1.0
  w  vs n     -> 1.0
  w  vs p     -> 1.0


## 3. Phase 1 — Learn OCR-mangled POS tokens

A candidate OCR-mangled POS is a short middle token in a 3-token string
like `"mengabaikan yr ngagungkan"` where the middle token (`yr`) is not a
canonical POS. We collect all such candidates across **all** dictionaries.

A repair `garbled → canonical` is accepted iff:
- it appears in **≥ 2 different dictionaries** (rules out one-off glitches),
- OCR-aware distance to the best POS is **≤ 2.0**,
- the gap to the second-best POS is **≥ 0.5** (rejects ambiguous ties).

Ambiguous cases are surfaced to `ocr_pos_ambiguous.csv` for manual review
rather than silently dropped.

In [12]:
def candidate_pos_token(form: str) -> Optional[str]:
    """Middle short token in a [word X word] pattern, where X is not a canonical POS."""
    if not isinstance(form, str):
        return None
    toks = form.strip().split()
    if len(toks) != 3:
        return None
    left, mid, right = toks
    if not (left.isalpha() or "-" in left):
        return None
    if not (right.isalpha() or "-" in right):
        return None
    if len(mid) > 4 or mid in POS_TAGS:
        return None
    if re.fullmatch(r"[A-Za-z]{1,4}\.?", mid) is None:
        return None
    return mid.rstrip(".")


def build_ocr_fixup_table(
    all_csvs: list[Path],
    min_support_dicts: int = 2,
    max_cost: float = 2.0,
    min_margin: float = 0.5,
) -> tuple[dict[str, str], list[dict]]:
    support: dict[str, set[str]] = defaultdict(set)
    for csv in all_csvs:
        dict_id = parse_dict_id(csv.name)
        if dict_id is None:
            continue
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue
        if "form" not in df.columns:
            continue
        for form in df["form"].dropna():
            c = candidate_pos_token(form)
            if c is not None:
                support[c].add(dict_id)

    fixups, ambiguous = {}, []
    for garbled, dicts in support.items():
        if len(dicts) < min_support_dicts:
            continue
        scored = sorted(
            ((pos, ocr_aware_distance(garbled, pos)) for pos in POS_TAGS),
            key=lambda x: x[1],
        )
        best_pos, best_cost = scored[0]
        second_cost = scored[1][1]
        if best_cost > max_cost:
            continue
        if (second_cost - best_cost) < min_margin:
            ambiguous.append({
                "garbled": garbled,
                "supporting_dicts": sorted(dicts),
                "best": f"{best_pos}({best_cost})",
                "second": f"{scored[1][0]}({second_cost})",
            })
            continue
        fixups[garbled] = best_pos
    return fixups, ambiguous

In [13]:
all_csvs = sorted(
    p for p in SRC_DIR.glob("*_Morphology.csv") if not p.name.startswith(".")
)
print(f"Found {len(all_csvs)} morphology CSVs")

ocr_fixups, ambiguous = build_ocr_fixup_table(all_csvs)

print(f"\nLearned {len(ocr_fixups)} OCR repair rules:")
for garbled, canon in sorted(ocr_fixups.items()):
    print(f"  {garbled!r:>8}  ->  {canon}")

if ambiguous:
    print(f"\n{len(ambiguous)} ambiguous candidates (NOT applied):")
    for a in ambiguous:
        print(
            f"  {a['garbled']!r:>8}  best={a['best']:<10} "
            f"second={a['second']:<10} in={a['supporting_dicts']}"
        )

# Persist for audit
pd.DataFrame(
    [{"garbled": g, "canonical": c} for g, c in sorted(ocr_fixups.items())]
).to_csv(FIXUP_REPORT, index=False)
if ambiguous:
    pd.DataFrame(ambiguous).to_csv(DST_DIR / "ocr_pos_ambiguous.csv", index=False)

Found 68 morphology CSVs

Learned 20 OCR repair rules:
     'ada'  ->  adv
     'bau'  ->  a
     'bun'  ->  num
     'cak'  ->  a
    'dadi'  ->  adv
      'di'  ->  vi
      'fl'  ->  vi
     'hal'  ->  a
      'ii'  ->  vi
      'il'  ->  vi
      'it'  ->  vt
     'itu'  ->  vt
      'ki'  ->  vi
      'li'  ->  vi
       'm'  ->  n
      'si'  ->  vi
      'ta'  ->  a
      'tt'  ->  vt
       'u'  ->  a
      'yr'  ->  vt

15 ambiguous candidates (NOT applied):
       'i'  best=vi(1.0)    second=v(1.0)     in=['10', '12', '18', '21', '24', '26', '27', '33', '34', '36', '41', '45', '46']
       'e'  best=v(1.0)     second=a(1.0)     in=['10', '77']
       'l'  best=v(1.0)     second=a(1.0)     in=['10', '35', '36', '45']
       'f'  best=v(1.0)     second=a(1.0)     in=['10', '45']
       't'  best=vt(1.0)    second=v(1.0)     in=['10', '12', '18', '24', '34', '46']
      'yg'  best=vt(1.5)    second=vi(1.5)    in=['11', '19']
       'c'  best=v(1.0)     second=a(1.0)     in=['18'

## 4. Phase 2 — Split forms using canonical + learned POS tokens

In [14]:
def split_form(form: str, ocr_fixups: dict[str, str]) -> tuple[str, str, str, bool]:
    """Return (left, tag, right, ocr_fixed). No direction assumed yet."""
    if not isinstance(form, str):
        return "", "", "", False
    form = form.strip()
    if not form:
        return "", "", "", False

    m = POS_SPLIT_RE.search(form)
    if m is not None:
        return (
            form[: m.start()].strip(),
            m.group(1),
            form[m.end():].strip(),
            False,
        )

    if ocr_fixups:
        alt = "|".join(re.escape(k) for k in ocr_fixups)
        ocr_re = re.compile(rf"(?:^|\s)({alt})(?:\s|$)")
        m2 = ocr_re.search(form)
        if m2 is not None:
            return (
                form[: m2.start()].strip(),
                ocr_fixups[m2.group(1)],
                form[m2.end():].strip(),
                True,
            )

    return form, "", "", False

## 5. Phase 3 — Process each dictionary

`LookupIsFromIndonesia.csv` tells us which side is Indonesian. The pipeline-
compatible `form` column always holds the derived form in the **same
language as `kata`** (the headword), matching the thesis Lampiran
convention:
- Dict 18 (Indonesian→Javanese): `kata=balon`, `form=membaloni`, `tag=vt`
- Dict 24 (Minangkabau→Indonesian): `kata=acuik`, `form=maangkek`, `tag=vt`

In [15]:
def process_one(
    csv_path: Path,
    ocr_fixups: dict[str, str],
    direction_lookup: dict[str, int],
) -> Optional[dict]:
    dict_id = parse_dict_id(csv_path.name)
    if dict_id is None:
        return None

    direction = direction_for(dict_id, direction_lookup)
    if direction is None:
        direction = 1  # default to Indonesian->Regional
        direction_known = False
    else:
        direction_known = True

    source_role, target_role = roles_for_morphology(direction)

    df = pd.read_csv(csv_path)
    if "form" not in df.columns:
        return None

    splits = df["form"].apply(lambda f: split_form(f, ocr_fixups))
    left = [s[0] for s in splits]
    inferred_tag = [s[1] for s in splits]
    right = [s[2] for s in splits]
    ocr_fixed = [s[3] for s in splits]

    existing_tag = (
        df.get("tag", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
    )
    final_tag = [
        e if e and e.lower() != "nan" else i
        for e, i in zip(existing_tag, inferred_tag)
    ]

    # Map left/right to ind/reg
    if source_role == "ind":
        form_ind, form_reg = left, right
    else:
        form_ind, form_reg = right, left

    audit = pd.DataFrame({
        "kata": df["kata"].values,
        "form_ind": form_ind,
        "form_reg": form_reg,
        "tag": final_tag,
        "ocr_pos_fixed": ocr_fixed,
        "form_raw": df["form"].values,
        "direction": direction,
        "direction_known": direction_known,
    })

    # Pipeline-compat: form is same language as `kata` (the source side)
    compat = pd.DataFrame({
        "kata": df["kata"].values,
        "form": form_ind if direction == 1 else form_reg,
        "tag": final_tag,
    })

    compat.to_csv(DST_DIR / f"{dict_id}_Morphology.csv", index=False)
    audit.to_csv(DST_DIR / f"{dict_id}_Morphology_audit.csv", index=False)

    return {
        "dict_id": dict_id,
        "direction": direction,
        "direction_known": direction_known,
        "rows": len(df),
        "tags_recovered": sum(1 for t in final_tag if t),
        "both_sides_split": sum(
            1 for li, re_ in zip(form_ind, form_reg) if li and re_
        ),
        "ocr_repaired": sum(ocr_fixed),
    }

In [16]:
direction_lookup = load_direction_lookup()
summaries = []
for csv in all_csvs:
    summary = process_one(csv, ocr_fixups, direction_lookup)
    if summary is not None:
        summaries.append(summary)

summary_df = pd.DataFrame(summaries).sort_values(
    "dict_id", key=lambda s: s.astype(int)
)
summary_df.to_csv(DST_DIR / "_summary.csv", index=False)
summary_df

,dict_id,direction,direction_known,rows,tags_recovered,both_sides_split,ocr_repaired
10,1,0,True,6,0,0,0
21,2,0,True,12,4,1,1
30,3,0,True,1,1,0,0
38,4,1,True,728,713,311,5
47,5,1,True,820,796,224,17
...,...,...,...,...,...,...,...
61,85,0,True,1,0,0,0
62,87,1,True,200,123,26,7
63,89,0,True,106,105,0,3
65,91,0,True,3668,3614,54,7


## 6. Totals

In [17]:
print(f"Dictionaries processed:       {len(summary_df)}")
print(f"Dictionaries w/o direction:   {(~summary_df['direction_known']).sum()}")
print(f"Total rows:                   {summary_df['rows'].sum()}")
print(f"Total tags recovered:         {summary_df['tags_recovered'].sum()}")
print(f"Total rows split both sides:  {summary_df['both_sides_split'].sum()}")
print(f"Total OCR-repaired POS:       {summary_df['ocr_repaired'].sum()}")
print(f"\nOutputs written to: {DST_DIR.resolve()}")

Dictionaries processed:       68
Dictionaries w/o direction:   0
Total rows:                   40245
Total tags recovered:         33739
Total rows split both sides:  5748
Total OCR-repaired POS:       733

Outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\10. Morphology - Fixed


## 7. Sanity checks

If `direction_known=False` for any dictionary, update
`LookupIsFromIndonesia.csv` and rerun — the default is
`direction=1` (Indonesian→Regional) which is a best guess, not a
verified fact.

In [18]:
unknown_direction = summary_df[~summary_df['direction_known']]
if len(unknown_direction) > 0:
    print("⚠ Dictionaries with unknown direction (defaulted to 1):")
    print(unknown_direction.to_string(index=False))
else:
    print("✓ All dictionaries have known direction from lookup table.")

✓ All dictionaries have known direction from lookup table.


## 8. Derived Bilingual Lexicon (new resource)

The audit files contain both Indonesian and regional derivations side-by-side.
This is richer than Billex (which only has lemma-level translations) because
it gives derivation-level bilingual pairs tagged with POS.

Example from dict 4 (Jambi):
- Billex has: `abai → lale` (lemma-level)
- DerivedBillex will have: `abai / mengabaikan → melalekan` (derivation-level, tagged `vt`)

This resource is genuinely new — it was buried in the unsplit `form` strings
until now and no other stage of the pipeline produces it.

In [19]:
derived_billex_rows = []

for csv in all_csvs:
    dict_id = parse_dict_id(csv.name)
    if dict_id is None:
        continue

    audit_path = DST_DIR / f"{dict_id}_Morphology_audit.csv"
    if not audit_path.exists():
        continue

    audit = pd.read_csv(audit_path)

    # Keep only rows where BOTH sides were successfully split
    valid = audit[
        audit["form_ind"].astype(str).str.strip().ne("")
        & audit["form_reg"].astype(str).str.strip().ne("")
    ].copy()

    if len(valid) == 0:
        continue

    direction = int(valid["direction"].iloc[0])

    # Convention: kata_asal is the source-language side of the source
    # dictionary (matches the Billex convention).
    #   dir=1 (Ind->Regional): kata_asal=Indonesian derivation,
    #                          kata_tujuan=Regional derivation
    #   dir=0 (Regional->Ind): kata_asal=Regional derivation,
    #                          kata_tujuan=Indonesian derivation
    if direction == 1:
        kata_asal = valid["form_ind"]
        kata_tujuan = valid["form_reg"]
    else:
        kata_asal = valid["form_reg"]
        kata_tujuan = valid["form_ind"]

    derived = pd.DataFrame({
        "dict_id": dict_id,
        "lemma": valid["kata"].values,         # the headword this derivation belongs to
        "kata_asal": kata_asal.values,         # source-language derivation
        "kata_tujuan": kata_tujuan.values,     # target-language derivation
        "tag": valid["tag"].values,
        "direction": direction,
    })

    # Per-dictionary file (drop dict_id/direction columns since they're
    # constant per file, but keep them in the combined output)
    per_dict = derived.drop(columns=["dict_id", "direction"])
    per_dict.to_csv(DST_DIR / f"{dict_id}_DerivedBillex.csv", index=False)

    derived_billex_rows.append(derived)

if derived_billex_rows:
    all_derived = pd.concat(derived_billex_rows, ignore_index=True)
    all_derived.to_csv(DST_DIR / "_DerivedBillex_all.csv", index=False)

    print(f"Derived bilingual pairs extracted: {len(all_derived):,}")
    print(f"Dictionaries contributing:         {all_derived['dict_id'].nunique()}")
    print(f"\nRows per dictionary:")
    print(all_derived.groupby("dict_id").size().sort_index())
    print(f"\nTag distribution:")
    print(all_derived["tag"].value_counts())
    print(f"\nSample (dict 4 / Jambi):")
    sample = all_derived[all_derived["dict_id"] == "4"].head(10)
    print(sample[["lemma", "kata_asal", "kata_tujuan", "tag"]].to_string(index=False))
else:
    print("No derived bilingual pairs extracted. Run sections 1–6 first to "
          "produce the _Morphology_audit.csv files.")

Derived bilingual pairs extracted: 40,245
Dictionaries contributing:         60

Rows per dictionary:
dict_id
1        6
10    1125
11     205
12    2645
13     143
14      93
15     155
16     471
17       4
18    2684
19    2032
2       12
21     482
23     693
24    2172
25     143
26    1014
27     612
28     183
29       2
3        1
33      27
34    2322
35     999
36     626
37       2
38      73
4      728
41    2883
42    1037
43    1444
45    1039
46    4621
5      820
50     252
51      39
54     728
55     422
56       1
57       3
58       9
60       7
61       2
62      63
63      53
66       6
67      65
68    2343
70       3
71      28
77     278
79     331
8      124
84       2
85       1
87     200
89     106
9        8
91    3668
94       5
dtype: int64

Tag distribution:
tag
v       17896
n       10574
a        2368
vt       1095
adv       816
vi        699
p         151
num       120
pron       20
Name: count, dtype: int64

Sample (dict 4 / Jambi):
lemma   kata_asa

## 9. Quality check on the new resource

Two sanity checks on the derived bilingual lexicon:

1. **Rows with a non-empty tag** — these are the high-confidence pairs.
2. **Derivation pairs where `kata_asal == kata_tujuan`** — either legitimate
   loanwords preserved across derivation (rare) or unhandled extraction
   noise (common). Flag for review.

In [20]:
if derived_billex_rows:
    tagged = all_derived[all_derived["tag"].astype(str).str.strip().ne("")]
    identical = all_derived[
        all_derived["kata_asal"].astype(str).str.lower()
        == all_derived["kata_tujuan"].astype(str).str.lower()
    ]

    print(f"Total derived pairs:        {len(all_derived):,}")
    print(f"Pairs with POS tag:         {len(tagged):,} ({len(tagged)/len(all_derived):.1%})")
    print(f"Identical asal == tujuan:   {len(identical):,} ({len(identical)/len(all_derived):.1%})")

    if len(identical) > 0:
        print(f"\nSample of identical pairs (review candidates):")
        print(identical.head(10)[["dict_id", "lemma", "kata_asal", "tag"]].to_string(index=False))

        identical.to_csv(DST_DIR / "_DerivedBillex_identical_review.csv", index=False)
        print(f"\nWritten to: {DST_DIR / '_DerivedBillex_identical_review.csv'}")

Total derived pairs:        40,245
Pairs with POS tag:         40,245 (100.0%)
Identical asal == tujuan:   75 (0.2%)

Sample of identical pairs (review candidates):
dict_id        lemma   kata_asal tag
     18      barakan      barang  vt
     18    gemb long      gembok  vt
     18 ku kur-kukur     kukusan  vt
     19        bubar membubarkan  vt
     19         loak       lobak   n
     19         maju   memajukan  vt
     19        pusat  memusatkan  vt
     19        rawat   perawatan   n
     19        restu    merestui  vt
     19        saiam   menyalami  vt

Written to: ..\Ekstraksi\10. Morphology - Fixed\_DerivedBillex_identical_review.csv
